# Notebook 22 — Vector Databases for RAG
Goal: Understand how a vector index replaces brute-force similarity search in a realistic RAG pipeline.

This notebook demonstrates the pattern:

`documents → chunking → embeddings → vector index → nearest-neighbor search → retrieval`

It uses a lightweight FAISS example to show how real retrieval systems store and search embeddings efficiently.

## Section 1 — Install and Import Libraries

In [ ]:
# If needed:
# pip install sentence-transformers faiss-cpu pandas numpy scikit-learn

import re
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## Section 2 — Example Documents
We will use a small scientific text corpus similar to the earlier RAG notebooks.

In [ ]:
documents = [
    """KRAS is a commonly mutated oncogene in colorectal cancer. KRAS mutations can drive abnormal signaling through the MAPK pathway. These alterations are often associated with resistance to certain targeted therapies.""",

    """EGFR is an important receptor tyrosine kinase in cancer biology. EGFR inhibitors are used in several tumor types, especially lung cancer. Clinical response can depend on mutation status and pathway context.""",

    """Organoid models are increasingly used in translational cancer research. These systems can capture patient-specific drug response patterns and may help bridge experimental assays and clinical outcomes.""",

    """TP53 is a tumor suppressor gene involved in genome stability, apoptosis, and cell-cycle regulation. TP53 mutations are among the most common events across human cancers."""
]

docs_df = pd.DataFrame({"document": documents})
docs_df

## Section 3 — Chunk the Documents

In [ ]:
def chunk_text(text, max_sentences=2):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = " ".join(sentences[i:i+max_sentences]).strip()
        if chunk:
            chunks.append(chunk)
    return chunks

all_chunks = []
for doc_id, doc in enumerate(documents):
    chunks = chunk_text(doc, max_sentences=2)
    for chunk_id, chunk in enumerate(chunks):
        all_chunks.append({
            "doc_id": doc_id,
            "chunk_id": chunk_id,
            "chunk_text": chunk
        })

chunks_df = pd.DataFrame(all_chunks)
chunks_df

## Section 4 — Create Embeddings

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = model.encode(chunks_df["chunk_text"].tolist())
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

chunk_embeddings.shape

## Section 5 — Baseline Retrieval with Cosine Similarity
This is the simple approach used in the earlier RAG notebooks.

In [ ]:
def retrieve_cosine(query, top_k=3):
    query_embedding = model.encode([query]).astype("float32")
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]
    ranked_idx = np.argsort(similarities)[::-1][:top_k]

    results = chunks_df.iloc[ranked_idx].copy()
    results["score"] = similarities[ranked_idx]
    return results

query = "Which gene is commonly mutated in colorectal cancer?"
retrieve_cosine(query, top_k=3)

## Section 6 — Build a FAISS Vector Index
FAISS stores embeddings in a searchable vector index.

For this simple demo we use `IndexFlatL2`, which performs exact search.
In larger systems, FAISS can also support approximate nearest-neighbor search.

In [ ]:
embedding_dim = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(embedding_dim)
index.add(chunk_embeddings)

index.ntotal

## Section 7 — Query the FAISS Index
Instead of computing cosine similarity against every chunk directly, we search the vector index.

In [ ]:
def retrieve_faiss(query, top_k=3):
    query_embedding = model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)

    results = chunks_df.iloc[indices[0]].copy()
    results["distance"] = distances[0]
    return results

retrieve_faiss(query, top_k=3)

## Section 8 — Compare Cosine vs FAISS Retrieval
Inspect whether the same chunks are returned and how the ranking behaves.

In [ ]:
cosine_results = retrieve_cosine(query, top_k=3)
faiss_results = retrieve_faiss(query, top_k=3)

cosine_results, faiss_results

## Section 9 — Why Vector Databases Matter
For tiny datasets, cosine similarity in memory is fine.

For real RAG systems with thousands or millions of chunks, vector indexes are used because they provide:

- fast nearest-neighbor search
- scalable retrieval
- efficient storage of embeddings
- the basis for systems such as FAISS, Chroma, Pinecone, Weaviate, and Milvus

## Section 10 — Wrap Retrieval into a Simple Function

In [ ]:
def rag_retrieve(query, method="faiss", top_k=3):
    if method == "faiss":
        return retrieve_faiss(query, top_k=top_k)
    elif method == "cosine":
        return retrieve_cosine(query, top_k=top_k)
    else:
        raise ValueError("method must be 'faiss' or 'cosine'")

rag_retrieve("What models capture patient-specific drug response?", method="faiss", top_k=3)

## Section 11 — Exercises

1. Try different queries and compare cosine vs FAISS retrieval.
2. Increase `top_k` and inspect how the rankings change.
3. Change the chunk size and rebuild the index.
4. Add a new scientific document and re-index the chunks.
5. Modify the wrapper function so it prints only chunk text and scores/distances.

## Skills Gained
- understanding how vector indexes are used in RAG
- building a FAISS index from embeddings
- querying embeddings with nearest-neighbor search
- comparing brute-force cosine retrieval with indexed retrieval
- understanding how real RAG systems scale beyond toy examples